In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')


In [2]:

train['is_train'] = 1
test['is_train'] = 0

test_temp = test.copy()
test_temp['label'] = np.nan 

all_data = pd.concat([train, test_temp], axis=0, ignore_index=True)

thread_stats = all_data.groupby('post_id').agg({
    'comment': 'count',              # How long is the thread? (Long threads = arguments)
    'upvote': ['mean', 'std', 'sum'], # Is the thread popular?
    'downvote': ['mean', 'std', 'sum'] # Is the thread hated/controversial?
})

thread_stats.columns = [
    'thread_vol', 
    'thread_up_mean', 'thread_up_std', 'thread_up_sum', 
    'thread_down_mean', 'thread_down_std', 'thread_down_sum'
]
thread_stats = thread_stats.reset_index()

thread_stats['thread_engagement'] = thread_stats['thread_up_sum'] + thread_stats['thread_down_sum']
thread_stats['thread_negativity_ratio'] = thread_stats['thread_down_sum'] / (thread_stats['thread_up_sum'] + 1)

train = train.merge(thread_stats, on='post_id', how='left')
test = test.merge(thread_stats, on='post_id', how='left')

fill_zero_cols = ['thread_up_std', 'thread_down_std']
train[fill_zero_cols] = train[fill_zero_cols].fillna(0)
test[fill_zero_cols] = test[fill_zero_cols].fillna(0)

print("Thread features added successfully.")

Thread features added successfully.


In [3]:
import re

def add_term_indicator_columns(
    df,
    terms,
    prefix,
    case_insensitive=True
):
    df = df.copy()

    flags = re.IGNORECASE if case_insensitive else 0

    for term in terms:
        col_name = f"{prefix}_{term}"
        pattern = re.escape(term)
        df[col_name] = df["comment"].str.contains(
            pattern,
            regex=True,
            flags=flags,
            na=False
        )

    return df


In [4]:
# === Feature Engineering and Preprocessing ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion


thread_features = [
    'thread_vol', 'thread_up_mean', 'thread_up_std', 
    'thread_down_mean', 'thread_down_std', 
    'thread_engagement', 'thread_negativity_ratio'
]

# Update X_full to include these new columns
# Note: We added 'post_id' to the list just in case, but we won't use it in the model directly
cols_to_use = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 
    'upvote', 'downvote', "if_1", "if_2", 
    "disability", "race", "religion", "gender", 
    'comment'
] + thread_features 

X_full = train[cols_to_use].copy()
y = train['label']

X_full["text_len"] = X_full["comment"].str.len()
X_full["word_count"] = X_full["comment"].str.split().str.len()

X_full["question_marks"] = X_full["comment"].str.count(r"\?")
X_full["exclamations"] = X_full["comment"].str.count(r"!")

X_full[["text_len", "word_count"]]  = X_full[["text_len", "word_count"]].fillna(0)
X_full[["question_marks", "exclamations",]] = X_full[["question_marks", "exclamations"]].fillna(False)

X_full = X_full.fillna('Null')

violent_terms = [
    "kill", "die", "death", "shoot", "fire", "killing", 
    "dead", "hang", "hit", "burn", "murder", "rid",
    "shot", "punch", "shooting", "guns", "suicide", "bomb",
    "gun", "fired", "kick", "knife", "killed", "torture",
    "bullet", "slap", "destroy", "stab"
]
X_full = add_term_indicator_columns(X_full,terms=violent_terms,prefix="v")

moral_terms = ["deserve", "should", "solution", "need to", "have to", "all of them"]
X_full = add_term_indicator_columns(X_full,terms=moral_terms,prefix="m")

group_terms = ["muslim", "jew", "jewish", "christian", "catholic", "women", "men", "female", "male",
                "black", "white", "asian", "immigrant", "refugee", "gay", "lesbian", "native", "indian"]

X_full = add_term_indicator_columns(X_full,terms=group_terms,prefix="g")

personal_attack = ["you", "your", "idiot", "stupid", "blind", "troll", "liar", "dumb", "moron"]

X_full = add_term_indicator_columns(X_full,terms=personal_attack,prefix="p")

base_num_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', "if_1", "if_2", 
    "disability", "text_len", "word_count", "question_marks", "exclamations"
    ] + thread_features

term_feature_cols = [
    c for c in X_full.columns
    if c.startswith(("v_", "m_", "g_", "p_"))
]

num_features = base_num_features + term_feature_cols
text_feature = "comment"
cat_features = ["race", "religion", "gender"]


feature_encoder = ColumnTransformer(
    transformers=[
        # ("text_vec", TfidfVectorizer(max_features=5000), text_feature),
        ("num_scaler", StandardScaler(), num_features),  # Fixed: wrapped in list
        ("cats", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
)

In [5]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.metrics import classification_report, f1_score


train_df = X_full[y.notna()].copy()
y_train = y[y.notna()].astype(int)

# Configuration
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Containers for predictions
oof_preds_lr = np.zeros((len(train_df), 4)) # 4 classes
test_preds_lr = np.zeros((len(test), 4))

vec = TfidfVectorizer(
    max_features=15000,           # Increased from 5k to capture rare violent words
    ngram_range=(1, 2),           # Word N-grams
    analyzer='char_wb',           # Character N-grams are robust to misspellings
    min_df=5
)

print("Vectorizing text for Level 1...")
X_text_train = vec.fit_transform(train_df['comment'])
X_text_test = vec.transform(test['comment'])

print("Training Level 1 (Logistic Regression)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_text_train, y_train)):
    # Split data
    X_tr, X_val = X_text_train[train_idx], X_text_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    clf = LogisticRegression(C=1.0, solver='sag', max_iter=1000, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    
    # Predict Probabilities
    val_probs = clf.predict_proba(X_val)
    oof_preds_lr[val_idx] = val_probs
    
    # Accumulate Test Predictions
    test_preds_lr += clf.predict_proba(X_text_test) / n_folds
    
    # Quick Check
    val_preds = np.argmax(val_probs, axis=1)
    f1 = f1_score(y_val, val_preds, average='macro')
    print(f"Fold {fold+1} LR F1: {f1:.4f}")
    print(classification_report(y_val, val_preds))


print("Level 1 Complete.")



Vectorizing text for Level 1...
Training Level 1 (Logistic Regression)...
Fold 1 LR F1: 0.4333
              precision    recall  f1-score   support

           0       0.66      0.88      0.75     22835
           1       0.68      0.28      0.40      3184
           2       0.60      0.37      0.45     12488
           3       0.58      0.07      0.13      1093

    accuracy                           0.65     39600
   macro avg       0.63      0.40      0.43     39600
weighted avg       0.64      0.65      0.61     39600

Fold 2 LR F1: 0.4304
              precision    recall  f1-score   support

           0       0.66      0.88      0.75     22835
           1       0.67      0.28      0.39      3183
           2       0.60      0.36      0.45     12488
           3       0.58      0.07      0.13      1094

    accuracy                           0.65     39600
   macro avg       0.63      0.40      0.43     39600
weighted avg       0.64      0.65      0.61     39600

Fold 3 LR F1: 

In [6]:
# 3. ADD PREDICTIONS TO DATAFRAME
# -------------------------------
# We add these 4 probability columns as new features for LightGBM
prob_cols = ['lr_prob_0', 'lr_prob_1', 'lr_prob_2', 'lr_prob_3']

train_df[prob_cols] = oof_preds_lr
test[prob_cols] = test_preds_lr



In [7]:
train_df.columns

Index(['emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', 'if_1',
       'if_2', 'disability', 'race', 'religion', 'gender', 'comment',
       'thread_vol', 'thread_up_mean', 'thread_up_std', 'thread_down_mean',
       'thread_down_std', 'thread_engagement', 'thread_negativity_ratio',
       'text_len', 'word_count', 'question_marks', 'exclamations', 'v_kill',
       'v_die', 'v_death', 'v_shoot', 'v_fire', 'v_killing', 'v_dead',
       'v_hang', 'v_hit', 'v_burn', 'v_murder', 'v_rid', 'v_shot', 'v_punch',
       'v_shooting', 'v_guns', 'v_suicide', 'v_bomb', 'v_gun', 'v_fired',
       'v_kick', 'v_knife', 'v_killed', 'v_torture', 'v_bullet', 'v_slap',
       'v_destroy', 'v_stab', 'm_deserve', 'm_should', 'm_solution',
       'm_need to', 'm_have to', 'm_all of them', 'g_muslim', 'g_jew',
       'g_jewish', 'g_christian', 'g_catholic', 'g_women', 'g_men', 'g_female',
       'g_male', 'g_black', 'g_white', 'g_asian', 'g_immigrant', 'g_refugee',
       'g_gay', 'g_lesbian',

In [8]:
test.columns

Index(['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment', 'is_train', 'thread_vol', 'thread_up_mean',
       'thread_up_std', 'thread_up_sum', 'thread_down_mean', 'thread_down_std',
       'thread_down_sum', 'thread_engagement', 'thread_negativity_ratio',
       'lr_prob_0', 'lr_prob_1', 'lr_prob_2', 'lr_prob_3'],
      dtype='object')

In [9]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import time


drop_cols = ['comment',] # Drop raw text and ID
X_lgb_train = train_df.drop(columns=drop_cols)
X_lgb_test = test.drop(columns=drop_cols)



# Train/validation split - stratified for balanced classes
X_train, X_valid, y_train, y_valid = train_test_split(
    X_lgb_train, y_train, test_size=0.2, random_state=0, stratify=y
)


modelA = Pipeline([
        ("prep", feature_encoder),
        ("model", lgb.LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=8,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        ))
    ])

modelA.fit(X_train,y_train)

print(classification_report(y_valid, modelA.predict(X_valid)))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.96      0.95      0.96     22835
           1       0.75      0.72      0.73      3183
           2       0.82      0.88      0.85     12488
           3       0.62      0.36      0.46      1094

    accuracy                           0.89     39600
   macro avg       0.79      0.73      0.75     39600
weighted avg       0.89      0.89      0.89     39600



In [10]:
import numpy as np

def predict_with_threshold(model, X, class_idx=3, threshold=0.2):
    probs = model.predict_proba(X)
    y_pred = np.argmax(probs, axis=1)
    y_pred[probs[:, class_idx] > threshold] = class_idx
    return y_pred


In [15]:
y_pred = predict_with_threshold(modelA, X_valid, class_idx=3, threshold=0.2)

print(classification_report(y_valid, y_pred))
print("Macro F1:", f1_score(y_valid, y_pred, average="macro"))
#LGBM SCORE

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.96      0.95      0.96     22835
           1       0.75      0.71      0.73      3183
           2       0.83      0.86      0.84     12488
           3       0.49      0.54      0.51      1094

    accuracy                           0.89     39600
   macro avg       0.76      0.76      0.76     39600
weighted avg       0.89      0.89      0.89     39600

Macro F1: 0.7605128917065871
